In [0]:
CATALOG = "workspace"
DATABASE = "qualite_eau"

In [0]:
df_bronze = spark.table(f"{CATALOG}.{DATABASE}.bronze_qualite_eau")
print(f"Bronze : {df_bronze.count():,} lignes")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def transform_silver(df):
    return (
        df
        # Explosion des arrays code_reseau et nom_reseau
        .withColumn("code_reseau", F.explode_outer("code_reseau"))
        .withColumn("nom_reseau", F.explode_outer("nom_reseau"))

        # Typage de la date
        .withColumn("date_prelevement", F.to_timestamp("date_prelevement"))

        # Extraction valeur numérique du résultat
        .withColumn("resultat_numerique",
            F.when(
                F.col("resultat_alphanumerique").rlike(r"^\d+$"),
                F.col("resultat_alphanumerique").cast(DoubleType())
            ).when(
                F.col("resultat_alphanumerique").rlike(r"^\d+[,.]\d+$"),
                F.regexp_replace("resultat_alphanumerique", ",", ".").cast(DoubleType())
            ).when(
                F.col("resultat_alphanumerique").rlike(r"^[<>]\d+[,.]\d+$"),
                F.regexp_replace(
                    F.regexp_extract("resultat_alphanumerique", r"[\d,.]+", 0),
                    ",", "."
                ).cast(DoubleType())
            ).otherwise(None))

        # Typage des seuils
        .withColumn("limite_qualite_num",
            F.expr("try_cast(regexp_replace(regexp_extract(limite_qualite_parametre, '(\\d+[.,]?\\d*)', 1), ',', '.') AS DOUBLE)"))
        .withColumn("reference_qualite_num",
            F.expr("try_cast(regexp_replace(regexp_extract(reference_qualite_parametre, '(\\d+[.,]?\\d*)', 1), ',', '.') AS DOUBLE)"))

        # Nettoyage textuel
        .withColumn("libelle_parametre", F.lower(F.trim("libelle_parametre")))
        .withColumn("nom_commune", F.initcap(F.trim("nom_commune")))

        # Catégorisation des paramètres
        .withColumn("categorie_parametre",
            F.when(F.col("libelle_parametre").rlike(
                r"(bact[eé]ri|escherichia|enteroco|coliform)"), "microbiologie")
            .when(F.col("libelle_parametre").rlike(
                r"(nitrate|nitrite|pesticide|aluminium|cuivre|fer|plomb|fluorure|chlorure|conductivit|^ph)"), "chimie")
            .when(F.col("libelle_parametre").rlike(
                r"(tritium|dose.*indicative|radioactiv)"), "radioactivite")
            .otherwise("autre"))

        # Indicateur de conformité
        .withColumn("conclusion_lower", 
            F.lower(F.col("conclusion_conformite_prelevement")))
        .withColumn("est_conforme",
            F.when(
                F.col("conclusion_lower").rlike(r"non.conforme.*(limite|exigence)"),
                False
            ).when(
                F.col("conclusion_lower").rlike(r"conforme aux (limites|exigences)"),
                True
            ).otherwise(None))
        .drop("conclusion_lower")

        # Enrichissements temporels
        .withColumn("annee", F.year("date_prelevement"))
        .withColumn("mois", F.month("date_prelevement"))
        .withColumn("trimestre", F.quarter("date_prelevement"))

        # Suppression des colonnes techniques Bronze
        .drop("_source_file", "_ingestion_timestamp", "_annee")

        # Filtres qualité
        .filter(F.col("date_prelevement").isNotNull())
        .filter(F.col("code_commune").isNotNull())

        # Suppression doublons
        .dropDuplicates(["code_commune", "date_prelevement", "code_parametre", "code_reseau"])
    )

df_silver = transform_silver(df_bronze)
print(f"Silver : {df_silver.count():,} lignes")

In [0]:
display(df_silver.limit(10))

In [0]:
display(df_silver.groupBy("categorie_parametre")
    .agg(
        F.count("*").alias("nb_mesures"),
        F.round(F.avg(F.col("est_conforme").cast("int")) * 100, 2).alias("pct_conforme")
    )
    .orderBy("categorie_parametre"))

In [0]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_departement", "annee")
    .saveAsTable(f"{CATALOG}.{DATABASE}.silver_qualite_eau"))

print("✅ Table Silver écrite")